# RAG-1 focuses on loading in PDF files (in this case, Apple 10-k files), and save them as text. #

This `notebook` focusses on the building the chunked dataset from the 20 year 10-k from APPLE. These documents have been stored in a google drive and will be available as PDFs. The idea is to `parse` the text in the pdf and save those chunks of text.

In [ ]:
#在google colab里面跑，不要在本地跑，除非你有本事自己搞个虚拟环境
#不要改任何东西，文件名和路径我已经改好了，理论上来说只要按住运行就行
#理论上来说rag1,rag2只用跑一遍，后面直接跑rag3就行
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q langchain langchain-community PyPDF2 langchain_openai nltk #不用管版本报错，应该没什么问题

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 449.6/449.6 kB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


Please `DONOT` delete the next cell. This will download a folder containing the data of pdfs.

In [ ]:
#Suppose you have a bunch of company files. Save them in a google drive folder. In our case, the folder ID is in the URL: https://drive.google.com/drive/folders/1yoJ0GF21QtRDpYdfzza7zYf2Z4BvqRbB?dmr=1&ec=wgc-drive-hero-goto
#如果出现access的报错就找我
!gdown --folder 1yoJ0GF21QtRDpYdfzza7zYf2Z4BvqRbB?dmr=1&ec=wgc-drive-hero-goto


Retrieving folder contents
Processing file 1LrnjQsL1RXiPB3xEyn3s-lyG_-25FEWC Nvidia_2016.pdf
Processing file 12yech0nVArhzGBfVKDJwoYDizqYegL-D Nvidia_2017.pdf
Processing file 1xC8rphTHQQtC51hZaCISV0I8COdcQlFa Nvidia_2018.pdf
Processing file 1SNFkoDraizSupAG04ctJQ8OBgbyruFU1 Nvidia_2019.pdf
Processing file 1_V6jkQi-Qs76Rtj8eFsfFTHfRaWsGEpv Nvidia_2020.pdf
Processing file 1xDvzYsH--LgChJDqch1wLCU9zr8YKrcm Nvidia_2021.pdf
Processing file 1Wm-8sclWvgEryCn35jyk_DFCdRrWFDhk Nvidia_2022.pdf
Processing file 1Imihl5skidxVAxBhzhbhVABrNhCFZWMh Nvidia_2023.pdf
Processing file 1n-KKLA5hkVFBDLgExZ4hLBEmXZk69IdM Nvidia_2024.pdf
Processing file 1SP6rlx1Fvrk2KkF1Y0RwujwuY0modpJy Nvidia_2025.pdf
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1LrnjQsL1RXiPB3xEyn3s-lyG_-25FEWC
To: /content/Nvidia-10k/Nvidia_2016.pdf
100% 839k/839k [00:00<00:00, 97.3MB/s]
Downloading...
From: https://drive.google.c

`from PyPDF2 import PdfReader` This imports the parser for the pdfs. We will use `NLTK` sentence tokenizer

In [ ]:
import glob
import os
!pip install PyPDF2 #这个库可能没有得装一下
from PyPDF2 import PdfReader
from nltk.tokenize import sent_tokenize
from tqdm import tqdm
import nltk
import pickle

nltk.download('punkt')
nltk.download('punkt_tab') #上面那个好像搞不到nltk,再跑一遍这个

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
filepaths = glob.glob("/content/Nvidia-10k/*.pdf") # change "Apple-10K" to the company you like to study
chunks_to_store = {}

In [ ]:
split_size = 15 ### Change this accordingly
overlap = 1 ### Change this accordingly
typedoc = "10-K"
company_name = "Nvidia"
ticker = "NVID"

In [ ]:
def load_from_text(fstarter : str, text: str, split_size : int, overlap : int):
    text = sent_tokenize(text)
    i = 0
    data = []
    while i < len(text):
        splits = [fstarter]
        splits.extend(text[i:i+split_size])
        data.append(" ".join(splits))
        i = i + (split_size - overlap)

    return data

In [ ]:
for i in tqdm(range(len(filepaths))):
  file_name = filepaths[i].split("/")[-1]
  try:
    # Attempt to extract year assuming the format CompanyName_Year.pdf
    filing_year = file_name.split("_")[1].split(".")[0]
  except IndexError:
    # If the above fails, try extracting year assuming the format CompanyNameYear.pdf
    filing_year = ''.join(filter(str.isdigit, file_name))

  pdf_reader = PdfReader(filepaths[i])

  datas = ''
  for i, page in enumerate(pdf_reader.pages):
    page_text = page.extract_text()
    if page_text:
      datas += page_text

  fstarter = f"Information from {typedoc} of {company_name}({ticker}) released in {filing_year}."
  chunked_data = load_from_text(fstarter, datas, split_size, overlap)

  chunks_to_store[filing_year] = chunked_data
#这里结果要显示100%才算成功了

100%|██████████| 10/10 [01:12<00:00,  7.24s/it]


In [ ]:
with open("chunked_dataset_Nvidia.pkl", "wb") as fp:   #change "Apple-10K" to the company you like to study
  pickle.dump(chunks_to_store, fp, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
!mkdir -p /content/drive/MyDrive/AccountingandFinance/

In [ ]:
!cp /content/chunked_dataset_Nvidia.pkl /content/drive/MyDrive/AccountingandFinance/